# Sprint 2: baseline TB classifier — train, choose thresholds, evaluate

Reproduces the Sprint 2 experiment in Colab against a pinned commit of the repository, so
the run can be tied to the exact code and data that produced it.

**What the baseline established**

- MobileNetV2 transfer learning on 559 training radiographs, validated on 121, with the
  120-image test split untouched until final evaluation.
- Four variants compared on validation alone (`ml/artifacts/experiments.csv`). Lighter
  augmentation with a deeper fine-tune won on both AUC and recall.
- Thresholds chosen from validation against the WHO target product profile for a TB triage
  test, rather than defaulting to 0.50.
- Test result: **AUC 0.891, sensitivity 0.949, specificity 0.508**, each with a bootstrap
  confidence interval.

**The two open problems this notebook is set up to attack**

1. **Specificity is 50.8%** against the WHO target of 70%. The model over-refers: 30 of 61
   healthy patients in the test set were flagged. This is the main defect.
2. **The Grad-CAM is not verified to look at lungs.** On one TB case it activated over the
   mediastinum and near the "PA ERECT" annotation marker. The model may be reading
   acquisition artefacts rather than pathology. Montgomery ships 276 manual lung masks, so
   this is measurable rather than arguable.

Training logic lives in the repository (`ml/preprocessing.py`, `ml/scripts/*.py`), not in
this notebook. The notebook orchestrates and records.

## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** (do this first; it restarts the session).
- All Sprint 2 code must be pushed to GitHub. The repository check below fails loudly if it
  is not, rather than silently running an older version.
- `datasets.zip` must be in Drive at `MyDrive/tb-screening-system/`. It is written by
  `scripts/backup_to_drive.py` on the laptop.
- Set the knobs in section 2. Everything else runs top to bottom.
- Colab already provides TensorFlow, scikit-learn, Pillow and matplotlib. Nothing is
  installed, so the session is not rebuilt; section 1 records the versions actually used.

## 1. Repository and environment

In [ ]:
%%bash
cd /content
if [ -d tb-screening-system ]; then
  cd tb-screening-system && git fetch -q origin && git checkout -q main && git pull -q origin main
else
  git clone -q https://github.com/derrickkimutai1/tb-screening-system.git
fi

In [ ]:
%cd /content/tb-screening-system

from pathlib import Path

required = [
    "ml/preprocessing.py",
    "ml/provenance.py",
    "ml/scripts/prepare_data.py",
    "ml/scripts/train_model.py",
    "ml/scripts/select_thresholds.py",
    "ml/scripts/evaluate_model.py",
    "ml/scripts/plot_training.py",
    "ml/artifacts/splits.csv",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, (
    f"Not in the cloned repository: {missing}. Commit and push from your machine, "
    "then rerun from the top."
)
print("Repository check passed.")

In [ ]:
import subprocess
import sys

import numpy as np
import sklearn
import tensorflow as tf

sys.path.insert(0, "ml")
import provenance  # noqa: E402

COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()

gpus = tf.config.list_physical_devices("GPU")
print("commit       ", COMMIT[:12])
print("tensorflow   ", tf.__version__)
print("keras        ", tf.keras.__version__)
print("scikit-learn ", sklearn.__version__)
print("numpy        ", np.__version__)
print(
    "GPU          ", gpus[0].name if gpus else "none - Runtime > Change runtime type > T4 GPU"
)
print()
print("dataset      ", provenance.dataset_fingerprint())

## 2. Configuration

In [ ]:
# --- Run identity -----------------------------------------------------------
RUN_TAG = "colab-baseline"  # names the checkpoint, metadata and Drive folder
SEED = 42  # matches ml/scripts/train_model.py

# --- What to do this session ------------------------------------------------
DO_TRAIN = True  # False reuses a model already saved in Drive
DO_THRESHOLDS = True  # choose decision/triage thresholds on validation
DO_EVALUATE = True  # final test evaluation; run once per model

# --- Training ---------------------------------------------------------------
AUGMENTATION = "light"  # "standard", "light" or "none"
EPOCHS = 30  # upper bound; early stopping decides the real end
FINE_TUNE_EPOCHS = 15
UNFREEZE = 60  # base layers unfrozen in stage 2
FINE_TUNE_LR = 1e-4  # 1e-5 was too timid; 1e-4 gave the best AUC but is unstable
BATCH_SIZE = 32

# --- Where results go -------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/tb-screening-system"

## 3. Google Drive

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_TAG}"
os.makedirs(RUN_DIR, exist_ok=True)
print("drive root :", DRIVE_ROOT)
print("run folder :", RUN_DIR)
print("contents   :", sorted(os.listdir(DRIVE_ROOT)))

## 4. Datasets

The images are copied from Drive as one archive and unpacked onto the Colab disk. Reading
800 files one at a time through the Drive mount is far slower than copying a single
archive and extracting it locally.

In [ ]:
import shutil
import zipfile

archive = f"{DRIVE_ROOT}/datasets.zip"
assert os.path.exists(archive), (
    "datasets.zip is not in Drive. Run scripts/backup_to_drive.py on the laptop first."
)

if not os.path.isdir("data/raw/shenzhen"):
    shutil.copy(archive, "/content/datasets.zip")
    with zipfile.ZipFile("/content/datasets.zip") as bundle:
        bundle.extractall(".")

montgomery = len(os.listdir("data/raw/montgomery/MontgomerySet/CXR_png"))
shenzhen = len(os.listdir("data/raw/shenzhen/CXR_png"))
print(f"montgomery {montgomery} files, shenzhen {shenzhen} files")

In [ ]:
import csv
from collections import Counter

with open("ml/artifacts/splits.csv", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

for split in ("train", "validation", "test"):
    counts = Counter(r["label"] for r in rows if r["split"] == split)
    total = counts["1"] + counts["0"]
    print(
        f"  {split:<11}{total:>4} images   {counts['1']:>3} TB   {counts['1'] / total:.1%} TB"
    )

# The split manifest is committed, so Colab and the laptop use the same one. Nothing is
# re-split here; prepare_data.py owns that and is seeded.

## 5. The baseline to beat

Two reference points, so an improvement can be recognised as one.

**Trivial baseline.** The classes are near balanced at 49.2% TB, so always predicting the
majority class scores about 50.8% accuracy and an AUC of 0.5. Any useful model must clearly
exceed that.

**Current model.** AUC 0.891, sensitivity 0.949, specificity 0.508. Sensitivity already
meets the WHO target of 90%; **specificity is the number to improve**, against a target of
70%. A change that raises specificity without dropping sensitivity below 0.90 is progress;
anything else is a trade, not an improvement.

In [ ]:
import json

reference = None
if os.path.exists("ml/artifacts/metrics.json"):
    with open("ml/artifacts/metrics.json", encoding="utf-8") as handle:
        reference = json.load(handle)
    print("committed baseline, test split")
    for name, value in reference["metrics"].items():
        low, high = reference["confidence_intervals_95"][name]
        print(f"  {name:<12}{value:.4f}   95% CI [{low:.3f}, {high:.3f}]")
else:
    print("No committed baseline found; this run establishes one.")

## 6. Train

Runs the repository's training script rather than redefining the model here, so what runs
in Colab is the same code that runs on the laptop. Augmentation is applied to the training
split only, and the script never loads the test split.

In [ ]:
if DO_TRAIN:
    !python ml/scripts/train_model.py \
        --tag {RUN_TAG} \
        --augmentation {AUGMENTATION} \
        --epochs {EPOCHS} \
        --fine-tune-epochs {FINE_TUNE_EPOCHS} \
        --unfreeze {UNFREEZE} \
        --fine-tune-lr {FINE_TUNE_LR} \
        --batch-size {BATCH_SIZE}
else:
    print("DO_TRAIN is False; reusing the model already in the repository or Drive.")

In [ ]:
if DO_TRAIN:
    !python ml/scripts/train_model.py --tag {RUN_TAG} --promote
    !python ml/scripts/plot_training.py --tag {RUN_TAG}

with open("ml/artifacts/experiments.csv", encoding="utf-8") as handle:
    for line in handle:
        print(line.rstrip())

## 7. Threshold selection

Chosen on validation only. The decision threshold is the lowest cutoff reaching 90%
sensitivity, the WHO target for a TB triage test; the high-suspicion threshold is the
lowest reaching 95% specificity, so that a high-suspicion label is worth prioritising.

In [ ]:
if DO_THRESHOLDS:
    !python ml/scripts/select_thresholds.py

with open("ml/models/metadata.json", encoding="utf-8") as handle:
    print(json.dumps(json.load(handle)["thresholds"], indent=2))

## 8. Evaluation on the held-out test split

Run **once** per model. Every figure carries a bootstrap confidence interval, because 120
images cannot support a single precise number.

In [ ]:
if DO_EVALUATE:
    !python ml/scripts/evaluate_model.py

with open("ml/artifacts/metrics.json", encoding="utf-8") as handle:
    current = json.load(handle)

print(f"{'metric':<13}{'baseline':>10}{'this run':>11}{'change':>10}")
for name, value in current["metrics"].items():
    if reference:
        was = reference["metrics"][name]
        print(f"  {name:<11}{was:>10.4f}{value:>11.4f}{value - was:>+10.4f}")
    else:
        print(f"  {name:<11}{'-':>10}{value:>11.4f}{'-':>10}")

## 9. Did the explanation move onto the lungs?

The open question from the baseline. Montgomery provides a left and right lung mask for
each of its 138 images, so the share of Grad-CAM activation falling inside the lung fields
can be measured rather than judged by eye. A model reading pathology should concentrate
most of its activation inside the lungs; one reading annotation markers or borders will not.

In [ ]:
!python ml/scripts/evaluate_explanations.py --split test

with open("ml/artifacts/gradcam_evaluation.json", encoding="utf-8") as handle:
    explanations = json.load(handle)

print()
print(f"images measured        {explanations['images']}")
print(f"activation in lungs    {explanations['share_in_lungs_mean']:.1%}")
print(f"lungs as image share   {explanations['lung_area_mean']:.1%}")
print(f"lift                   {explanations['lift_mean']:.2f}")
print(f"  on TB cases          {explanations['lift_tb_mean']}")
print(f"  on normal cases      {explanations['lift_normal_mean']}")
print()
print(explanations["interpretation"])

## 10. Save the run summary to Drive

In [ ]:
summary = {
    "run_tag": RUN_TAG,
    "git_commit": COMMIT,
    "provenance": provenance.describe(),
    "ran_on": "colab",
    "gpu": gpus[0].name if gpus else None,
    "versions": {
        "tensorflow": tf.__version__,
        "keras": tf.keras.__version__,
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__,
    },
    "configuration": {
        "seed": SEED,
        "augmentation": AUGMENTATION,
        "epochs": EPOCHS,
        "fine_tune_epochs": FINE_TUNE_EPOCHS,
        "unfreeze": UNFREEZE,
        "fine_tune_lr": FINE_TUNE_LR,
        "batch_size": BATCH_SIZE,
    },
    "thresholds": json.load(open("ml/models/metadata.json", encoding="utf-8"))["thresholds"],
    "test_metrics": current["metrics"],
    "confidence_intervals_95": current["confidence_intervals_95"],
    "confusion_matrix": current["confusion_matrix"],
    "explanations": {
        "lift_mean": explanations["lift_mean"],
        "share_in_lungs_mean": explanations["share_in_lungs_mean"],
        "lung_area_mean": explanations["lung_area_mean"],
        "interpretation": explanations["interpretation"],
    },
}

with open(f"{RUN_DIR}/run_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, default=str)

for artefact in (
    "ml/models/tb_mobilenetv2_v1.keras",
    "ml/models/metadata.json",
    "ml/artifacts/metrics.json",
    "ml/artifacts/experiments.csv",
    "ml/artifacts/test_predictions.csv",
    f"ml/artifacts/training_history_{RUN_TAG}.png",
    f"ml/artifacts/model_report_{RUN_TAG}.md",
    "ml/artifacts/confusion_matrix.png",
    "ml/artifacts/roc_curve.png",
    "ml/artifacts/threshold_selection.png",
    "ml/artifacts/gradcam_evaluation.json",
    "ml/artifacts/gradcam_lung_concentration.png",
):
    if os.path.exists(artefact):
        shutil.copy2(artefact, f"{RUN_DIR}/{os.path.basename(artefact)}")

print("saved to", RUN_DIR)
for name in sorted(os.listdir(RUN_DIR)):
    print("   ", name)

## Sprint 2 exit checklist

- [ ] Repository check passed, so the run used pushed code and the commit is recorded.
- [ ] Split counts match the committed manifest; the test split was not reloaded or re-split.
- [ ] Training used the repository scripts, with augmentation on the training split only.
- [ ] Thresholds chosen on validation against the WHO target, not defaulted to 0.50.
- [ ] Test evaluation run **once**, with confidence intervals.
- [ ] **Specificity compared against the 0.508 baseline** — this is the metric Sprint 3 must move.
- [ ] Grad-CAM lung share measured and compared against the lung area share.
- [ ] `run_summary.json` and all artefacts saved to the run folder in Drive.
- [ ] Notebook saved back to GitHub, so the recorded code matches the experiment.

**If specificity has not improved:** the threshold is doing what it was asked to do, so the
model is the limit, not the cutoff. The next controlled change is stronger regularisation —
higher dropout and weight decay — judged on validation only, since the training-to-validation
AUC gap of 0.047 says the model is fitting the 559 training images more closely than it
generalises.

**If the Grad-CAM lung share is near the lung area share:** the explanation is not evidence
of pathology and must not be presented as if it were. Masking the image borders and the
annotation markers before training is the controlled change to test.